**Note:** `export ANYWIDGET_HMR=1` before starting JupyterLab to enable hot module reloading for the widget code.

In [1]:
%reload_ext autoreload
%autoreload 2

In [2]:
import numpy as np
import pyviewarr

First, generate some example data in a 4D array and take an example image from it:

In [3]:
n_cubes, n_planes = 3, 2
coords = np.linspace(0, 2 * np.pi)
xx, yy = np.meshgrid(coords, coords)
cubes = []
for i in range(1, n_cubes + 1):
    planes = []
    for j in range(1, n_planes + 1):
        planes.append(np.sin(xx * i) * np.cos(yy * j))
    cubes.append(planes)
example_data = np.asarray(cubes)
example_image = example_data[0, 0]

To create an interactive viewer, use `pyviewarr.show()`:

In [4]:
pyviewarr.show(example_image)

You can specify the dimensions of the viewer in the call to `show()`:

In [5]:
pyviewarr.show(example_image, width=500, height=400)

All the options you can pass to `show` can be found on the `pyviewarr.ViewerConfig` object:

In [6]:
help(pyviewarr.ViewerConfig.__init__)

Help on function __init__ in module pyviewarr:

__init__(
    self,
    *,
    contrast: Optional[float] = None,
    bias: Optional[float] = None,
    stretch: Optional[Literal['linear', 'log', 'symmetric']] = None,
    zoom: Optional[float] = None,
    cmap: Optional[str] = None,
    colormap_reversed: Optional[bool] = None,
    vmin: Optional[float] = None,
    vmax: Optional[float] = None,
    xlim: Optional[Tuple[float, float]] = None,
    ylim: Optional[Tuple[float, float]] = None,
    rotation: Optional[float] = None,
    pivot: Optional[Tuple[float, float]] = None,
    show_pivot_marker: Optional[bool] = None
) -> None
    Initialize self.  See help(type(self)) for accurate signature.



You can specify `vmin=` and `vmax=` and they work like they do with matplotlib's `imshow()`. `xlim` and `ylim` should also be familiar from matplotlib.

In [7]:
pyviewarr.show(
    example_image,
    vmin=-0.5, vmax=0.5,
    cmap='RdYlBu',
    stretch='symmetric',
    width=500, height=400
)

Note that `cmap=` will only work with the colormaps included for that stretch mode:

* `stretch='linear'`/`'log'` — `'gray'`, `'inferno'`, `'magma'`
* `stretch='symmetric'` — `'RdBu'`, `'RdYlBu'`

Also: these are just strings, not full matplotlib colormap objects.

Occasionally, you may want to get values back out of the viewer, such as its configuration (so you can recreate it later). 

To do that, you need to:

1. Create the viewer
2. Set the array it should show
3. Display the viewer widget

In [8]:
widget = pyviewarr.create_viewer()
widget.set_array(example_image)
widget

If `widget` is the last line in a cell, it'll get displayed. Otherwise, you'll need to `from IPython.display import display` and call `display(widget)`.

Avoid `display()`ing the same widget multiple times, as this leads to unpredictable behavior and confusion about which one is "live".

Here we make a second widget instance to illustrate the other display method.

In [9]:
from IPython.display import display
widget2 = pyviewarr.create_viewer()
widget2.set_array(example_image)
widget2
display(widget2)

Having a reference to `widget` means you can do other stuff too:

In [10]:
widget.get_viewer_config()

ViewerConfig(contrast=1.0, bias=0.5, stretch='linear', zoom=1.0, cmap='gray', colormap_reversed=False, vmin=0.0, vmax=1.0, xlim=(0.0, 0.0), ylim=(0.0, 0.0), rotation=0.0, pivot=(0.0, 0.0), show_pivot_marker=False)

You can even update the array it should show after the fact (that is, in a later cell):

In [11]:
for w in [widget, widget2]:
    w.set_array(example_data[0, 1])

Since it's so useful, there's a helper like `show()` that returns the widget instance. Note, however, that it won't display automatically if you assign the return value to a variable.

In [12]:
widget3 = pyviewarr.viewarr(example_image)

If you don't need to refer back to the widget, just make `viewarr()` the last line of the cell:

In [13]:
pyviewarr.viewarr(example_image)

In [14]:
# Create a test image with a gradient and some features
arr = np.zeros((256, 256), dtype=np.float32)
# Add gradient
arr += np.linspace(0, 100, 256)
# Add a circle
y, x = np.ogrid[:256, :256]
circle_mask = ((x - 128)**2 + (y - 128)**2) < 50**2
arr[circle_mask] = 200
# Add some noise
arr += np.random.randn(256, 256) * 5

In [15]:
# Display the array
pyviewarr.show(arr)

When working with data cubes, it is common to visualize one slice at a time. The viewer widget handles this by adding a slice control above the image viewer. You can navigate through the slices and it will wrap around to the beginning if you reach the end.

In [16]:
# Test 3D array
arr_3d = np.random.rand(10, 256, 256).astype(np.float32)
pyviewarr.show(example_data[0])
example_data[0].shape

(2, 50, 50)

This even works with N-dimensional arrays, by adding more slice controls. In keeping with NumPy / matplotlib convention, the last two dimensions are the image and the preceding dimensions are slice-able.

In [17]:
pyviewarr.show(example_data)